# Ensemble Learning Lab: Bagging, Boosting, and Stacking

This notebook implements Bagging, Boosting, and Stacking classifiers on the Kaggle Playground Series S6E2 dataset.

In [ ]:

import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, AdaBoostClassifier,
                               GradientBoostingClassifier, RandomForestClassifier,
                               StackingClassifier)
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


In [ ]:

# Load data (adjust target column name if needed)
df = pd.read_csv("train.csv")

target = 'target'
X = df.drop(columns=[target])
y = df[target]

X = pd.get_dummies(X)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)


In [ ]:

results = []

def evaluate(model, name):
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)[:,1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, preds),
        "F1": f1_score(y_val, preds),
        "ROC_AUC": roc_auc_score(y_val, probs),
        "Train_Time": train_time
    })


In [ ]:

# Bagging
bagging = BaggingClassifier(
    base_estimator=DecisionTreeClassifier(),
    n_estimators=100,
    bootstrap=True,
    random_state=42
)
evaluate(bagging, "Bagging")


In [ ]:

# AdaBoost
ada = AdaBoostClassifier(
    base_estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)
evaluate(ada, "AdaBoost")


In [ ]:

# Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
)
evaluate(gb, "Gradient Boosting")


In [ ]:

# XGBoost
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)
evaluate(xgb, "XGBoost")


In [ ]:

# Stacking
estimators = [
    ('lr', LogisticRegression(max_iter=1000)),
    ('rf', RandomForestClassifier(n_estimators=200)),
    ('gb', GradientBoostingClassifier())
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000)
)
evaluate(stack, "Stacking")


In [ ]:

# Results comparison
results_df = pd.DataFrame(results)
results_df



## Analysis & Discussion

1. Boosting methods typically outperform bagging due to bias reduction.
2. Bagging reduces variance, boosting reduces bias.
3. Stacking may outperform individual models by combining strengths.
4. XGBoost is often preferred for deployment due to performance.
5. Boosting models are computationally more expensive than bagging.
